# Python 并发编程基础完整指南

> 涵盖多线程（threading）、多进程（multiprocessing）、协程（asyncio）


## 一、并发编程概览

### 三种并发模型的对比

| 模型 | 模块 | 适用场景 | 特点 |
|------|------|---------|------|
| **多线程** | threading | I/O 密集型 | 共享内存，GIL 限制，不能真正并行 |
| **多进程** | multiprocessing | CPU 密集型 | 独立内存，真正并行，开销大 |
| **协程** | asyncio | 高并发 I/O | 单线程，事件循环，非阻塞 |

### GIL（全局解释器锁）

> Python 的 GIL 保证同一时刻只有一个线程执行 Python 字节码。
> - 多线程：**不能**利用多核 CPU（I/O 阻塞时切换）
> - 多进程：**可以**利用多核 CPU（每个进程独立 GIL）
> - 协程：**单线程**，无 GIL 问题，适合海量连接

## 二、多线程（threading）

> 适合 **I/O 密集型** 任务：网络请求、文件读写、数据库操作

### 1. 基础用法：创建线程

In [ ]:
import threading
import time

def worker(name, delay):
    print(f"[{name}] 开始工作")
    time.sleep(delay)  # 模拟 I/O 操作
    print(f"[{name}] 完成工作，耗时 {delay}秒")

# 创建线程
thread1 = threading.Thread(target=worker, args=("线程A", 2))
thread2 = threading.Thread(target=worker, args=("线程B", 1))

# 启动线程
start = time.time()
thread1.start()
thread2.start()

# 等待线程完成
thread1.join()
thread2.join()

print(f"\n总耗时: {time.time() - start:.2f}秒")
# 输出约 2 秒（不是 3 秒，因为并行执行）

### 2. 线程池：ThreadPoolExecutor（推荐）

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def fetch_url(url):
    """模拟网络请求"""
    time.sleep(1)
    return f"数据来自 {url}"

urls = ["url1", "url2", "url3", "url4", "url5"]

# 使用线程池
start = time.time()
with ThreadPoolExecutor(max_workers=3) as executor:
    # 方式1：map（按顺序返回结果）
    # results = executor.map(fetch_url, urls)
    # for result in results:
    #     print(result)

    # 方式2：submit（异步提交，谁先完成谁先处理）
    futures = {executor.submit(fetch_url, url): url for url in urls}
    for future in as_completed(futures):
        url = futures[future]
        try:
            result = future.result()
            print(f"{url}: {result}")
        except Exception as e:
            print(f"{url} 出错: {e}")

print(f"\n总耗时: {time.time() - start:.2f}秒")
# 5个任务，3个线程，约需 2 秒（不是 5 秒）

### 3. 线程同步：锁（Lock）

In [ ]:
import threading

# 共享资源
counter = 0
lock = threading.Lock()

def increment():
    global counter
    for _ in range(100000):
        # 方式1：显式加锁解锁
        # lock.acquire()
        # counter += 1
        # lock.release()

        # 方式2：使用 with 语句（推荐）
        with lock:
            counter += 1

threads = []
for i in range(5):
    t = threading.Thread(target=increment)
    threads.append(t)
    t.start()

for t in threads:
    t.join()

print(f"最终计数: {counter}")  # 应该是 500000

# 如果没有锁，结果会小于 500000（数据竞争）

### 4. 其他同步原语

In [ ]:
import threading

# ========== RLock（可重入锁）==========
# 同一线程可以多次获取，不会死锁
rlock = threading.RLock()

def outer():
    with rlock:
        print("外层获取锁")
        inner()  # 同一线程再次获取锁

def inner():
    with rlock:
        print("内层获取锁")

outer()

# ========== Semaphore（信号量）==========
# 限制同时访问的线程数量
semaphore = threading.Semaphore(2)  # 最多 2 个线程同时执行

def limited_worker(name):
    with semaphore:
        print(f"{name} 开始执行")
        time.sleep(1)
        print(f"{name} 执行完毕")

# ========== Event（事件）==========
# 线程间通信
event = threading.Event()

def waiter():
    print("等待事件...")
    event.wait()  # 阻塞直到事件被设置
    print("事件已触发！")

def trigger():
    time.sleep(2)
    print("设置事件")
    event.set()

threading.Thread(target=waiter).start()
threading.Thread(target=trigger).start()

time.sleep(3)  # 等待演示完成

## 三、多进程（multiprocessing）

> 适合 **CPU 密集型** 任务：计算、图像处理、数据分析
> 每个进程有独立的 Python 解释器和 GIL，可以真正并行

### 1. 基础用法：创建进程

In [ ]:
import multiprocessing
import time

def cpu_intensive(n):
    """CPU 密集型任务"""
    count = 0
    for i in range(n):
        count += i ** 2
    return count

# 串行执行
start = time.time()
result1 = cpu_intensive(10_000_000)
result2 = cpu_intensive(10_000_000)
serial_time = time.time() - start
print(f"串行执行: {serial_time:.2f}秒")

# 并行执行（多进程）
start = time.time()
p1 = multiprocessing.Process(target=cpu_intensive, args=(10_000_000,))
p2 = multiprocessing.Process(target=cpu_intensive, args=(10_000_000,))
p1.start()
p2.start()
p1.join()
p2.join()
parallel_time = time.time() - start
print(f"并行执行: {parallel_time:.2f}秒")

print(f"\n加速比: {serial_time / parallel_time:.2f}x")

### 2. 进程池：Pool / ProcessPoolExecutor

In [ ]:
from multiprocessing import Pool
import time

def square(n):
    return n ** 2

numbers = list(range(1, 11))

# 方式1：使用 Pool
start = time.time()
with Pool(processes=4) as pool:
    # map：按顺序返回
    results = pool.map(square, numbers)
    print(f"map 结果: {results}")

    # apply_async：异步提交单个任务
    result = pool.apply_async(square, (20,))
    print(f"apply_async 结果: {result.get()}")

    # 使用列表推导批量提交
    async_results = [pool.apply_async(square, (n,)) for n in numbers]
    results2 = [r.get() for r in async_results]
    print(f"批量 async 结果: {results2}")

print(f"\n耗时: {time.time() - start:.2f}秒")

### 3. 进程间通信（IPC）

In [ ]:
from multiprocessing import Process, Queue, Pipe, Value, Array
import time

# ========== Queue（队列）==========
def producer(queue):
    for i in range(5):
        queue.put(f"产品 {i}")
        print(f"生产: 产品 {i}")
        time.sleep(0.1)

def consumer(queue):
    for _ in range(5):
        item = queue.get()
        print(f"消费: {item}")
        time.sleep(0.2)

queue = Queue()
p = Process(target=producer, args=(queue,))
c = Process(target=consumer, args=(queue,))
p.start()
c.start()
p.join()
c.join()

print("\n--- Queue 演示完成 ---")

# ========== Pipe（管道，双向通信）==========
def sender(conn):
    conn.send("Hello from sender")
    conn.close()

def receiver(conn):
    msg = conn.recv()
    print(f"收到消息: {msg}")
    conn.close()

parent_conn, child_conn = Pipe()
p1 = Process(target=sender, args=(child_conn,))
p2 = Process(target=receiver, args=(parent_conn,))
p1.start()
p2.start()
p1.join()
p2.join()

print("\n--- Pipe 演示完成 ---")

# ========== 共享内存（Value / Array）==========
# 注意：需要加锁保证安全
counter = Value('i', 0)  # 'i' 表示整数

def increment_shared(counter):
    for _ in range(1000):
        with counter.get_lock():  # 必须加锁
            counter.value += 1

processes = [Process(target=increment_shared, args=(counter,)) for _ in range(10)]
for p in processes:
    p.start()
for p in processes:
    p.join()

print(f"\n共享计数器最终值: {counter.value}")  # 应该是 10000

## 四、协程（asyncio）

> 适合 **高并发 I/O 密集型** 场景：Web 服务器、爬虫、实时通信
> 单线程 + 事件循环，比多线程更轻量，可管理数万连接

### 1. 基础概念

```
协程核心概念：
├── async def      → 定义协程函数
├── await          → 挂起协程，等待异步操作完成
├── asyncio.run()  → 运行主协程
├── asyncio.gather() → 并发执行多个协程
└── 事件循环        → 调度协程的执行
```

**关键点**：`await` 时协程挂起，事件循环去执行其他协程，I/O 完成后恢复。

### 2. 第一个协程

In [ ]:
import asyncio

async def say_hello():
    print("Hello")
    await asyncio.sleep(1)  # 模拟异步 I/O，不会阻塞其他协程
    print("World")

async def say_hi():
    print("Hi")
    await asyncio.sleep(0.5)
    print("There")

async def main():
    # 串行执行
    print("=== 串行执行 ===")
    await say_hello()
    await say_hi()

    print("\n=== 并发执行 ===")
    # 并发执行（使用 gather）
    await asyncio.gather(say_hello(), say_hi())

# 运行协程
asyncio.run(main())

### 3. 并发执行多个任务

In [ ]:
import asyncio
import time

async def fetch_data(url, delay):
    """模拟异步网络请求"""
    print(f"开始请求 {url}")
    await asyncio.sleep(delay)  # 非阻塞等待
    print(f"完成请求 {url}，耗时 {delay}秒")
    return f"{url} 的数据"

async def main():
    urls = [
        ("https://api1.com", 2),
        ("https://api2.com", 1),
        ("https://api3.com", 3),
    ]

    # 方式1：asyncio.gather（推荐）
    start = time.time()
    results = await asyncio.gather(
        *[fetch_data(url, delay) for url, delay in urls]
    )
    print(f"\ngather 结果: {results}")
    print(f"总耗时: {time.time() - start:.2f}秒")  # 约 3 秒（不是 6 秒）

    print("\n---")

    # 方式2：asyncio.create_task（更灵活）
    start = time.time()
    tasks = [asyncio.create_task(fetch_data(url, delay)) for url, delay in urls]

    # 可以在这里做其他事...

    # 等待所有任务完成
    results = await asyncio.gather(*tasks)
    print(f"\ntask 结果: {results}")
    print(f"总耗时: {time.time() - start:.2f}秒")

asyncio.run(main())

### 4. 超时控制与异常处理

In [ ]:
import asyncio

async def slow_task():
    await asyncio.sleep(5)
    return "完成"

async def error_task():
    await asyncio.sleep(1)
    raise ValueError("出错了！")

async def main():
    # 超时控制
    try:
        result = await asyncio.wait_for(slow_task(), timeout=2)
        print(f"结果: {result}")
    except asyncio.TimeoutError:
        print("任务超时！")

    # 异常处理
    try:
        await error_task()
    except ValueError as e:
        print(f"捕获异常: {e}")

    # 等待多个任务，返回已完成和未完成的
    tasks = [
        asyncio.create_task(asyncio.sleep(1)),
        asyncio.create_task(asyncio.sleep(2)),
        asyncio.create_task(asyncio.sleep(3)),
    ]

    done, pending = await asyncio.wait(tasks, timeout=1.5)
    print(f"\n已完成: {len(done)} 个")
    print(f"未完成: {len(pending)} 个")

    # 取消未完成的任务
    for task in pending:
        task.cancel()

asyncio.run(main())

### 5. 异步迭代器与上下文管理器

In [ ]:
import asyncio

# 异步迭代器
class AsyncCounter:
    def __init__(self, limit):
        self.limit = limit
        self.current = 0

    def __aiter__(self):
        return self

    async def __anext__(self):
        if self.current >= self.limit:
            raise StopAsyncIteration
        await asyncio.sleep(0.1)  # 模拟异步操作
        self.current += 1
        return self.current

async def main():
    print("异步迭代:")
    async for num in AsyncCounter(5):
        print(f"  {num}")

    # 异步上下文管理器
    class AsyncResource:
        async def __aenter__(self):
            print("异步获取资源...")
            await asyncio.sleep(0.5)
            print("资源已获取")
            return self

        async def __aexit__(self, exc_type, exc_val, exc_tb):
            print("异步释放资源...")
            await asyncio.sleep(0.5)
            print("资源已释放")
            return False

    print("\n异步上下文管理器:")
    async with AsyncResource() as res:
        print("使用资源中...")

asyncio.run(main())

## 五、三种模型对比实战

### 场景：下载 10 个网页

In [ ]:
import time

# 模拟下载函数（I/O 密集型）
def download_sync(url):
    time.sleep(1)  # 模拟网络延迟
    return f"{url} 的内容"

# ========== 1. 串行执行 ==========
urls = [f"https://example.com/page{i}" for i in range(1, 11)]

start = time.time()
results = [download_sync(url) for url in urls]
serial_time = time.time() - start
print(f"串行: {serial_time:.2f}秒")

# ========== 2. 多线程 ==========
import threading
from concurrent.futures import ThreadPoolExecutor

start = time.time()
with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(download_sync, urls))
thread_time = time.time() - start
print(f"多线程: {thread_time:.2f}秒")

# ========== 3. 多进程（不推荐用于 I/O 密集型）==========
from multiprocessing import Pool

start = time.time()
with Pool(processes=5) as pool:
    results = pool.map(download_sync, urls)
process_time = time.time() - start
print(f"多进程: {process_time:.2f}秒")

# ========== 4. 协程（推荐用于 I/O 密集型）==========
import asyncio

async def download_async(url):
    await asyncio.sleep(1)
    return f"{url} 的内容"

async def main():
    start = time.time()
    tasks = [download_async(url) for url in urls]
    results = await asyncio.gather(*tasks)
    return time.time() - start

async_time = asyncio.run(main())
print(f"协程: {async_time:.2f}秒")

print(f"\n=== 对比 ===")
print(f"串行 vs 多线程: {serial_time / thread_time:.1f}x")
print(f"串行 vs 协程: {serial_time / async_time:.1f}x")

### 场景：计算密集型任务

In [ ]:
import time

def cpu_task(n):
    """CPU 密集型：计算大量平方和"""
    count = 0
    for i in range(n):
        count += i ** 2
    return count

numbers = [5_000_000] * 4

# ========== 1. 串行 ==========
start = time.time()
results = [cpu_task(n) for n in numbers]
serial_time = time.time() - start
print(f"串行: {serial_time:.2f}秒")

# ========== 2. 多线程（GIL 限制，不会更快）==========
from concurrent.futures import ThreadPoolExecutor

start = time.time()
with ThreadPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(cpu_task, numbers))
thread_time = time.time() - start
print(f"多线程: {thread_time:.2f}秒")

# ========== 3. 多进程（真正并行）==========
from multiprocessing import Pool

start = time.time()
with Pool(processes=4) as pool:
    results = pool.map(cpu_task, numbers)
process_time = time.time() - start
print(f"多进程: {process_time:.2f}秒")

# ========== 4. 协程（不适合 CPU 密集型）==========
import asyncio

async def cpu_task_async(n):
    # 注意：协程中不能直接用 CPU 密集型代码，会阻塞事件循环
    # 需要用 run_in_executor 放到线程/进程中执行
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(None, cpu_task, n)

async def main():
    start = time.time()
    tasks = [cpu_task_async(n) for n in numbers]
    results = await asyncio.gather(*tasks)
    return time.time() - start

async_time = asyncio.run(main())
print(f"协程(用executor): {async_time:.2f}秒")

print(f"\n=== 对比 ===")
print(f"串行 vs 多进程: {serial_time / process_time:.1f}x")
print(f"多线程 vs 多进程: {thread_time / process_time:.1f}x")

## 六、选择指南

```
任务类型判断
    │
    ├── I/O 密集型？（网络、文件、数据库）
    │       │
    │       ├── 并发量高（>1000）？
    │       │       └── asyncio 协程 ✅
    │       │
    │       └── 并发量低，或需要兼容同步代码？
    │               └── threading 多线程 ✅
    │
    └── CPU 密集型？（计算、图像处理）
            └── multiprocessing 多进程 ✅
```

### 快速决策表

| 场景 | 推荐方案 | 原因 |
|------|---------|------|
| Web 爬虫（大量请求） | asyncio | 单线程管理数万连接 |
| 批量文件下载 | asyncio / threading | I/O 等待时切换 |
| 数据处理/机器学习 | multiprocessing | 利用多核 CPU |
| Web 服务器 | asyncio | 高并发低延迟 |
| 定时任务/后台作业 | threading | 简单够用 |
| 混合场景 | asyncio + ProcessPoolExecutor | I/O 用协程，CPU 用进程池 |

## 七、知识图谱总结

```
Python 并发编程
│
├── 多线程 threading
│   ├── Thread(target=func)
│   ├── ThreadPoolExecutor（推荐）
│   ├── 同步原语
│   │   ├── Lock / RLock
│   │   ├── Semaphore
│   │   ├── Event
│   │   └── Condition
│   └── 适用：I/O 密集型，GIL 限制不能并行
│
├── 多进程 multiprocessing
│   ├── Process(target=func)
│   ├── Pool / ProcessPoolExecutor
│   ├── 进程间通信
│   │   ├── Queue
│   │   ├── Pipe
│   │   └── Value / Array（共享内存）
│   └── 适用：CPU 密集型，真正并行
│
└── 协程 asyncio
    ├── async def / await
    ├── asyncio.run()
    ├── asyncio.gather()（并发）
    ├── asyncio.create_task()
    ├── 异步原语
    │   ├── async for（异步迭代）
    │   └── async with（异步上下文管理）
    └── 适用：高并发 I/O，单线程海量连接
```

### 关键要点速查

| 主题 | 核心要点 |
|------|---------|
| **GIL** | 多线程不能并行，多进程可以 |
| **threading** | 用 ThreadPoolExecutor，注意加锁 |
| **multiprocessing** | 用 Pool 或 ProcessPoolExecutor，注意 IPC |
| **asyncio** | 事件循环调度，await 挂起不阻塞 |
| **选择原则** | I/O 用 asyncio/threading，CPU 用 multiprocessing |
| **混合使用** | asyncio 中 CPU 任务用 `run_in_executor` |